# Predição: Prova de Fogo - V1

Autor:  Viviane da Rosa Sommer

Atualizado: 29/11/2024

## Objetivo:
Notebook para fazer a predição do modelo de Coral-Sol (MSO-V1) no vídeo da prova de fogo - V1.
Essa versão gera ao final os plots a cada 100 frames, junto com a máscara da predição para comparação.
No final, também é gerado o gráfico Predição (binária) vs Tempo.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch

import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import torchvision
import torch
from typing import List

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
RESIZED_DIM_CORAL = 800

coral_model = YOLO('runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')

## Funções Necessárias

In [ ]:
def process_results(results: list) -> any:
    """
    Processes the results obtained from the model, returning the first valid result with masks.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result containing masks, or None if no valid result is found.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None


def prediction_coral(img: np.ndarray) -> tuple:
    """
    Predicts coral objects using the coral model and returns a binary mask for coral regions.

    Parameters:
        img (np.ndarray): Input image for prediction.

    Returns:
        tuple: A tuple containing:
            - mask (np.ndarray): The binary mask of the predicted coral regions.
            - torch.Tensor: The binary mask as a PyTorch tensor.
            - str: The predicted class ("Positive" if the mask is not empty, "Negative" otherwise).
    """
    original_height, original_width = img.shape[:2]
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)
    
    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_boxes = boxes[coral_indices]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_boxes) > 0:
            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            coral_boxes = coral_boxes[nms_indices]
            coral_masks = coral_masks[nms_indices]
            final_coral_mask = torch.any(coral_masks, dim=0).int() * 255
        else:
            final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    
    final_coral_mask_np = final_coral_mask.cpu().numpy().astype(np.uint8)
    predicted_class = "Positive" if np.any(final_coral_mask_np) else "Negative"

    return final_coral_mask_np, torch.from_numpy(final_coral_mask_np).int(), predicted_class


def process_video(video_path: str) -> None:
    """
    Process the video frame by frame, perform inference and plot the frames with predictions.

    Parameters:
        video_path (str): Path to the video file.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error opening video.")
        return
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = 0
    time_stamps = []
    coral_found = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % 100 == 0:
            time_stamps.append(frame_count / fps)
            cropped_frame, top, bottom = crop_frame(frame)
            mask, _, predicted_class = prediction_coral(cropped_frame)
            frame_with_prediction = annotate_frame(cropped_frame, predicted_class, frame_count)
            plot_frame(frame_with_prediction, mask, frame, top, bottom, frame_count, predicted_class)
            coral_found.append(1 if predicted_class == "Positive" else 0)
        frame_count += 1
    
    cap.release()
    
    plot_coral_detection(time_stamps, coral_found)


def crop_frame(frame: np.ndarray) -> tuple:
    """
    Crop the frame to focus on the central region.

    Parameters:
        frame (np.ndarray): The frame to crop.

    Returns:
        tuple: A tuple containing the cropped frame and the top and bottom crop coordinates.
    """
    height, width, _ = frame.shape
    mid = height // 2
    top = max(0, mid - int(0.34 * height))
    bottom = min(height, mid + int(0.34 * height))
    cropped_frame = frame[top:bottom, :]
    return cropped_frame, top, bottom


def annotate_frame(frame: np.ndarray, annotation: str, frame_count: int) -> np.ndarray:
    """
    Annotate the frame with the prediction and crop it.

    Parameters:
        frame (np.ndarray): The video frame to annotate.
        annotation (str): The predicted class to annotate on the frame.
        frame_count (int): The current frame count.

    Returns:
        np.ndarray: The annotated and cropped frame.
    """
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(frame, f'Frame {frame_count}: Predicted class - {annotation}', 
                (10, 30), font, 1, (255, 0, 255), 2, cv2.LINE_AA)
    return frame


def plot_frame(cropped_frame: np.ndarray, mask: np.ndarray, original_frame: np.ndarray, top: int, bottom: int, frame_count: int, predicted_class: str) -> None:
    """
    Plot the given frame and mask using matplotlib.

    Parameters:
        cropped_frame (np.ndarray): The cropped frame to plot.
        mask (np.ndarray): The mask to plot.
        original_frame (np.ndarray): The original frame from the video.
        top (int): The top crop coordinate.
        bottom (int): The bottom crop coordinate.
        frame_count (int): The current frame count.
        predicted_class (str): The predicted class.
    """
    mask_resized = cv2.resize(mask, (original_frame.shape[1], bottom - top))
    mask_rgb = cv2.cvtColor(mask_resized, cv2.COLOR_GRAY2RGB)

    full_mask = np.zeros_like(original_frame)
    full_mask[top:bottom, :] = mask_rgb

    cropped_frame_rgb = cv2.cvtColor(cropped_frame, cv2.COLOR_BGR2RGB)
    full_mask_rgb = cv2.cvtColor(full_mask, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    axes[0].imshow(cropped_frame_rgb)
    axes[0].set_title(f'Cropped Frame with Binary Prediction\nFrame {frame_count}: Predicted class - {predicted_class}')
    axes[0].axis('off')
    
    axes[1].imshow(full_mask_rgb)
    axes[1].set_title('Coral Mask - Segmentation Prediction')
    axes[1].axis('off')
    
    plt.show()


def plot_coral_detection(time_stamps: List[int], coral_found: List[int]) -> None:
    """
    Plot the coral detection over time.

    Parameters:
        time_stamps (List[int]): List of time stamps corresponding to the frames.
        coral_found (List[int]): List of binary values indicating presence of coral.
    """
    plt.figure(figsize=(12, 6))
    plt.plot(time_stamps, coral_found, marker='o', linestyle='-', color='b')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Coral Found (1=Yes, 0=No)')
    plt.title('Coral Detection Over Time')
    plt.grid(True)
    plt.show()

## Prova de Fogo - Predição no vídeo

In [ ]:
video_path = 'Prova de fogo 1.mp4'
process_video(video_path)